# 1.2 Option B (Optional): Run Models Directly in a GPU DevLab

> **When to use this notebook:** Only run this notebook if you chose **Option B** and started a **GPU DevLab (JupyterLab on a GPU VM)** such as an L40S (`gpu-l40s-a`, `1gpu-8vcpu-32gb`). If you are working on your laptop with an Endpoint, use `endpoint_quickstart.ipynb` (Option A) instead.

Because the model runs inside this VM's GPU memory, we can measure internal model load time (`load_s`), inspect peak VRAM (`torch.cuda.max_memory_allocated()`), and iterate on prompts, seeds, and steps without reloading weights.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "generation.json").is_file())
sys.path.insert(0, str(CODE.resolve()))

# Store Hugging Face weights on the DevLab local disk:
os.environ.setdefault("HF_HOME", str(Path.home() / "hf"))

## 1. Check GPU and Install Diffusers Dependencies

Before loading weights, verify that CUDA is available and check how much free VRAM the GPU currently has. Remember that other running Jupyter kernels hold onto their GPU memory until shut down.

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.39.0", "transformers==5.13.1", "accelerate==1.14.0",
    "sentencepiece==0.2.1", "protobuf==6.33.0",
])
import torch

assert torch.cuda.is_available(), "CUDA is not available. Run this notebook on a GPU DevLab."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"], text=True))

## 2. Load the Pinned Sana Pipeline Once and Measure `load_s`

We load `Efficient-Large-Model/Sana_1600M_1024px_diffusers` at commit `ac0da2ff55fbe434795be0dce883042e4d49e2fc`. Keep this kernel alive while you experiment: restarting the kernel drops the pipeline from VRAM and re-pays the load time.

In [ ]:
from common import config
from generate import load_pipeline

cfg = config(CODE / "generation.json")
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()
pipe = load_pipeline(cfg)
torch.cuda.synchronize()
load_s = time.perf_counter() - t0
print(f"Loaded {cfg['model']} in {load_s:.2f} s | VRAM after load: {torch.cuda.max_memory_allocated() / 1e9:.3f} GB")

## 3. Generate and Watch Peak VRAM and Latency

Now that the model is in GPU memory, generating a `1024x1024` image takes only a few seconds (`4.42 s` and `10.91 GB` peak VRAM on an L40S). You can re-run this cell with a different seed or step count without reloading the model.

In [ ]:
from IPython.display import Image, display

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()
img = pipe(
    prompt=cfg["prompt"],
    **cfg["params"],
    generator=torch.Generator(device="cuda").manual_seed(cfg["seed"]),
).images[0]
torch.cuda.synchronize()
gen_s = time.perf_counter() - t0
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

preview_file = Path("runs/devlab_preview.png")
preview_file.parent.mkdir(parents=True, exist_ok=True)
img.save(preview_file)
print(f"Generation time: {gen_s:.2f} s | Peak VRAM: {peak_vram_gb:.3f} GB")
display(Image(filename=str(preview_file)))

## 4. Record a 3-Sample Run (`metrics.json`) and Compute Cost

`generate.run()` executes the full load + 3 sequential generations and writes `asset-000.png`..`asset-002.png`, `metrics.json`, and `trace.jsonl`.

In [ ]:
from common import calculate_cost
from generate import run

del pipe
torch.cuda.empty_cache()

folder = run(CODE / "generation.json", n=3, mode="jupyterlab")
report = json.loads((folder / "metrics.json").read_text())
print(json.dumps(report, indent=2))

cost_summary = calculate_cost(
    startup_s=report.get("startup_s") or 180.0,
    ready_s=report["load_s"],
    latency_s=report["latency_s"],
    price_per_hour=1.5484,
    counts=(1, 10, 100),
    api_price=0.04,
)
print("Cost projection:", json.dumps(cost_summary, indent=2))

## 5. Download Results and Stop the DevLab

Download your generated PNGs and `metrics.json` to your laptop, then stop or delete the DevLab in the Nebius console. Stopping stops GPU compute charges (though persistent disk still bills until deleted).

In [ ]:
print("ACTION REQUIRED: Download your files from", folder, "and stop/delete the DevLab in the console.")